# Prototype Starter

This notebook provides a starter workflow to:

1. Configure job ranks per Slurm script in `data/slurm_scripts`
2. Submit those scripts with `sbatch` (dry-run by default)
3. Load MPR performance data from `data/all_model_data.xlsx`
4. Run threaded MPR-INT and map final bids to per-job target frequency


In [ ]:
from pathlib import Path
import math
import shlex
import shutil
import subprocess

import pandas as pd
from mpr_int import run_threaded_mpr_int, normalize_job_perf_data

REPO_ROOT = Path.cwd()
SLURM_DIR = REPO_ROOT / "data" / "slurm_scripts"
PERF_XLSX = REPO_ROOT / "data" / "all_model_data.xlsx"

# Define job ranks here (key = script name without "run_" prefix / ".slurm").
# Example: "hpcg": 4 means run_hpcg.slurm submits with -n 4.
JOB_RANKS = {
    "comd": 1,
    "hpccg": 2,
    "hpcg": 4,
    "minife": 2,
    "minimd": 2,
    "xsbenchmpi": 1,
}

# Performance workbook sheet mapping per job (defaults to the same name).
PERF_SHEET_BY_JOB = {
    "xsbenchmpi": "xsbench",
}

CPUS_PER_RANK = 10
RANKS_PER_NODE = 2

SBATCH_PARTITION = "debug"
SBATCH_TIME = "00:30:00"
SBATCH_NODELIST = None  # Example: "ridlserver[04,11]"
SBATCH_EXCLUDE = None   # Example: "ridlserver02"

DRY_RUN = True  # Set to False to actually submit jobs.

print("Slurm dir:", SLURM_DIR)
print("Perf workbook:", PERF_XLSX)
print("Configured job ranks:", JOB_RANKS)


In [ ]:
def _build_sbatch_cmd(
    script_path,
    ranks,
    cpus_per_rank=10,
    ranks_per_node=2,
    partition="debug",
    time_limit="00:30:00",
    nodelist=None,
    exclude=None,
):
    if ranks <= 0:
        raise ValueError(f"ranks must be > 0 for {script_path.name}")
    if ranks_per_node <= 0:
        raise ValueError("ranks_per_node must be > 0")

    nodes = max(1, math.ceil(ranks / ranks_per_node))
    ntasks_per_node = min(ranks, ranks_per_node)

    cmd = [
        "sbatch",
        "-p", partition,
        "-N", str(nodes),
        "-n", str(ranks),
        "--ntasks-per-node", str(ntasks_per_node),
        "-c", str(cpus_per_rank),
        "-t", time_limit,
    ]
    if nodelist:
        cmd += ["--nodelist", nodelist]
    if exclude:
        cmd += ["--exclude", exclude]
    cmd.append(str(script_path))
    return cmd, nodes, ntasks_per_node


def submit_ranked_slurm_jobs(
    job_ranks,
    slurm_dir,
    cpus_per_rank=10,
    ranks_per_node=2,
    partition="debug",
    time_limit="00:30:00",
    nodelist=None,
    exclude=None,
    dry_run=True,
):
    if not dry_run and shutil.which("sbatch") is None:
        raise RuntimeError("sbatch not found in PATH. Use DRY_RUN=True for preview.")

    rows = []
    for job_name, ranks in job_ranks.items():
        script_path = slurm_dir / f"run_{job_name}.slurm"

        if not script_path.exists():
            rows.append({
                "job": job_name,
                "script": str(script_path),
                "ranks": int(ranks),
                "status": "MISSING_SCRIPT",
                "command": "",
                "message": "script not found",
            })
            continue

        cmd, nodes, ntasks_per_node = _build_sbatch_cmd(
            script_path=script_path,
            ranks=int(ranks),
            cpus_per_rank=cpus_per_rank,
            ranks_per_node=ranks_per_node,
            partition=partition,
            time_limit=time_limit,
            nodelist=nodelist,
            exclude=exclude,
        )
        cmd_str = " ".join(shlex.quote(t) for t in cmd)

        if dry_run:
            rows.append({
                "job": job_name,
                "script": str(script_path),
                "ranks": int(ranks),
                "nodes": int(nodes),
                "ntasks_per_node": int(ntasks_per_node),
                "status": "DRY_RUN",
                "command": cmd_str,
                "message": "preview only",
            })
            print(f"[DRY-RUN] {job_name}: {cmd_str}")
            continue

        proc = subprocess.run(cmd, text=True, capture_output=True, check=False)
        status = "SUBMITTED" if proc.returncode == 0 else "FAILED"
        rows.append({
            "job": job_name,
            "script": str(script_path),
            "ranks": int(ranks),
            "nodes": int(nodes),
            "ntasks_per_node": int(ntasks_per_node),
            "status": status,
            "command": cmd_str,
            "stdout": proc.stdout.strip(),
            "stderr": proc.stderr.strip(),
            "returncode": proc.returncode,
        })
        print(f"[{status}] {job_name}: {proc.stdout.strip() or proc.stderr.strip()}")

    return pd.DataFrame(rows)


submit_df = submit_ranked_slurm_jobs(
    job_ranks=JOB_RANKS,
    slurm_dir=SLURM_DIR,
    cpus_per_rank=CPUS_PER_RANK,
    ranks_per_node=RANKS_PER_NODE,
    partition=SBATCH_PARTITION,
    time_limit=SBATCH_TIME,
    nodelist=SBATCH_NODELIST,
    exclude=SBATCH_EXCLUDE,
    dry_run=DRY_RUN,
)
submit_df


In [ ]:
REQUIRED_COLS = ["Resource Reduction", "Extra Execution", "Power"]


def load_perf_data_for_jobs(xlsx_path, job_names, sheet_map=None):
    sheet_map = sheet_map or {}
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Workbook not found: {xlsx_path}")

    xl = pd.ExcelFile(xlsx_path)
    available = set(xl.sheet_names)

    jobs = {}
    audit_rows = []

    for job_name in job_names:
        sheet = sheet_map.get(job_name, job_name)

        if sheet not in available:
            audit_rows.append({
                "job": job_name,
                "sheet": sheet,
                "status": "MISSING_SHEET",
            })
            continue

        raw = pd.read_excel(xlsx_path, sheet_name=sheet)
        missing = [c for c in REQUIRED_COLS if c not in raw.columns]
        if missing:
            audit_rows.append({
                "job": job_name,
                "sheet": sheet,
                "status": "MISSING_COLUMNS",
                "details": ",".join(missing),
            })
            continue

        df = raw[REQUIRED_COLS].dropna().reset_index(drop=True)
        if df.empty:
            audit_rows.append({
                "job": job_name,
                "sheet": sheet,
                "status": "EMPTY_AFTER_CLEAN",
            })
            continue

        jobs[job_name] = df
        audit_rows.append({
            "job": job_name,
            "sheet": sheet,
            "status": "LOADED",
            "rows": len(df),
        })

    return jobs, pd.DataFrame(audit_rows)


job_perf_data, perf_audit_df = load_perf_data_for_jobs(
    xlsx_path=PERF_XLSX,
    job_names=list(JOB_RANKS.keys()),
    sheet_map=PERF_SHEET_BY_JOB,
)

print("Loaded job models:", list(job_perf_data.keys()))
perf_audit_df


In [ ]:
if not job_perf_data:
    raise RuntimeError("No valid job performance data loaded from all_model_data.xlsx")

target_reduction_w = 700.0

result = run_threaded_mpr_int(
    job_perf_data=job_perf_data,
    target_reduction_w=target_reduction_w,
    q_bounds=(0.1, 5.0),
    max_iters=100,
    delta_q_tol=0.05,
    residual_abs_tol=5.0,
    cycle_q_tol=0.05,
    host="127.0.0.1",
    port=8765,
    verbose=True,
)

summary = {
    "converged": result["converged"],
    "convergence_mode": result["convergence_mode"],
    "final_q": result["final_q"],
    "final_reduction_w": result["final_reduction_w"],
    "final_residual_w": result["final_residual_w"],
    "negotiation_time_s": result["negotiation_time_s"],
}
summary


In [ ]:
MAX_FREQ_MHZ = 2400.0

models = normalize_job_perf_data(job_perf_data)
delta_max_by_job = {m.client_id: float(m.delta_max) for m in models}
q_final = float(result["final_q"])

job_reduction = {}
job_freq_mhz = {}
for client_id, bid in result["final_bids"].items():
    delta = max(delta_max_by_job[client_id] - float(bid) / q_final, 0.0)
    delta = min(delta, 1.0)
    job_reduction[client_id] = delta
    job_freq_mhz[client_id] = MAX_FREQ_MHZ * (1.0 - delta)

freq_df = pd.DataFrame({
    "job": list(job_freq_mhz.keys()),
    "reduction_fraction": [job_reduction[k] for k in job_freq_mhz.keys()],
    "target_freq_mhz": [job_freq_mhz[k] for k in job_freq_mhz.keys()],
    "target_freq_ghz": [job_freq_mhz[k] / 1000.0 for k in job_freq_mhz.keys()],
}).sort_values("job").reset_index(drop=True)

freq_df
